mkdir llm-zoomcamp-hw2 && cd llm-zoomcamp-hw2
uv init --no-workspace
uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
uv add --dev huggingface-hub jupyter
uv run python -m ipykernel install --user --name llm-zoomcamp-hw2 --display-name "llm-zoomcamp-hw2"

PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed
wget $PREFIX/download.py
wget $PREFIX/embedder.py

uv run python download.py

## Q1. Embedding a query
Embed the following query:

*How does approximate nearest neighbor search work?*

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

-0.31       
-0.02       
0.12        
0.44        

In [6]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v = embed.encode(q1)

2026-06-28 13:11:14.529103854 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [7]:
v[0]

np.float64(-0.02058203437252893)

## Loading the data

We pull the lesson pages from the course repository, the same way as in homework 1. We pin to commit 8c1834d so everyone works with the same data.

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
documents[0]
# len(documents)

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

0.07        
0.37        
0.68        
0.92

In [8]:
# Find the specific document
doc = next(d for d in documents if d['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md')

# Embed its content
v_doc = embed.encode(doc['content'])

# Cosine similarity (dot product of normalized vectors = cosine similarity)
similarity = v.dot(v_doc)
print(similarity)

0.36107026789538205


## Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

In [ ]:
# from gitsource import chunk_documents
# chunks = chunk_documents(documents, size=2000, step=1000)

We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

In [ ]:
# scores = X.dot(v)

Which file does the highest-scoring chunk belong to (its filename)?

* 02-vector-search/lessons/03-embeddings-dataset.md     
* 02-vector-search/lessons/06-rag-vector.md     
* 02-vector-search/lessons/07-sqlitesearch-vector.md        
* 02-vector-search/lessons/09-onnx-embedder.md

In [ ]:
# len(chunks)
# chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [14]:
import numpy as np
from tqdm.auto import tqdm
from gitsource import chunk_documents

# 1. Chunk all documents
chunks = chunk_documents(documents, size=2000, step=1000)

# 2. Embed every chunk in batches
batch_size = 50
all_vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i + batch_size]
    texts = [chunk['content'] for chunk in batch]
    batch_vectors = embed.encode_batch(texts)
    all_vectors.extend(batch_vectors)

# 3. Stack into a matrix
X = np.array(all_vectors)

# 4. Score against Q1 query vector
scores = X.dot(v)

# 5. Find the chunk
idx = np.argmax(scores)
print("Score:   ", scores[idx])
print("Filename:", chunks[idx]['filename'])
print("Content: ", chunks[idx]['content'][:200])  # first 200 chars as a preview

  0%|          | 0/6 [00:00<?, ?it/s]

Score:    0.6489016436447387
Filename: 02-vector-search/lessons/07-sqlitesearch-vector.md
Content:  rch. We score
the query against every document and pick the top ones. It always finds
the true top matches, but it pays for that by touching everything.

Approximate nearest neighbor (ANN) search take


## Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

*What metric do we use to evaluate a search engine?*

Which file is the filename of the first result?

* 02-vector-search/lessons/04-vector-search.md      
* 04-evaluation/lessons/05-search-metrics.md        
* 04-evaluation/lessons/13-llm-as-judge.md      
* 05-monitoring/lessons/04-metrics.md       

In [ ]:
from tqdm.auto import tqdm
from gitsource import chunk_documents

# 1. Chunk all documents
chunks = chunk_documents(documents, size=2000, step=1000)

# 2. Embed every chunk in batches
batch_size = 50
all_vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i + batch_size]
    texts = [chunk['content'] for chunk in batch]
    batch_vectors = embed.encode_batch(texts)
    all_vectors.extend(batch_vectors)

# turn them into 2 dimensional array(matrix) where rows are documents(vectors) and columns are dimensions of the vectors
import numpy as np
X = np.array(all_vectors)

# index our documents and vectors
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["content"])
vindex.fit(X, chunks)

# searching
from embedder import Embedder

embed = Embedder()

query = "What metric do we use to evaluate a search engine?"

query_vector = embed.encode(q1)

results = vindex.search(query_vector, num_results=5)
results

## Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

* 02-vector-search/lessons/01-intro.md      
* 02-vector-search/lessons/02-embeddings.md     
* 02-vector-search/lessons/08-pgvector.md       
* 03-orchestration/lessons/05-rag.md